In [43]:
#### imports and loading of cancer PAR train qrels, train queries, corpus, LLM prompt

import srsly
import re
import pandas as pd
import os
import json
from collections import defaultdict
from openai import OpenAI, APIConnectionError
from pydantic import BaseModel
from typing import List
from pydantic import BaseModel, Field

In [6]:

# sep='\t' to remove the tab spaces
train_qrels = pd.read_csv("../Task4_Cancer_Patients/Updated_PAR_cancer/new_PAR_qrels/new_cancer_qrels_train.tsv", sep='\t')                         #table showing which patients (query) matches with which article (corpus)

patient_info = pd.DataFrame(list(srsly.read_jsonl("../Task4_Cancer_Patients/Updated_PAR_cancer/new_PAR_queries/new_cancer_train_queries.jsonl")))    #contains the cancer patient id's and their patient info

articles = pd.DataFrame(list(srsly.read_jsonl("../Task4_Cancer_Patients/Updated_PAR_cancer/new_cancer_corpus.jsonl")))                               #contains the article id's, titles, and abstracts 

print("train_qrels:", train_qrels.shape)
print("patient_info:", patient_info.shape)
print("articles:", articles.shape)

display(train_qrels.head())
display(patient_info.head())
display(articles.head())

with open("hippo_questions_prompt.txt", "r", encoding="utf-8") as f:
    prompt_template = f.read()


train_qrels: (742718, 3)
patient_info: (54810, 2)
articles: (398228, 3)


,query_id,corpus_id,score
0,8674405-1,29510273,1
1,8674405-1,10953134,1
2,8674405-1,30651078,1
3,8674405-1,7860418,1
4,8674405-1,21555684,1


,_id,text
0,6325026-2,A 57-year-old man was admitted to the emergenc...
1,6925471-1,A 17-year-old Syrian girl presented to our hos...
2,5223534-1,The patient presented as a 53 years old white ...
3,6982515-2,A 53-year-old man presented with a PSA level o...
4,4427095-1,A 35-year-old male presented with a swelling i...


,_id,title,text
0,16801861,Importance of biopsy of new bone lesions in pa...,Development of destructive bone lesions in a p...
1,17261432,Survival on dialysis post-kidney transplant fa...,A substantial number of patients return to dia...
2,15473586,Pericarditis following permanent pacemaker ins...,The appearance of pericarditis following inser...
3,1379294,Primary intraosseous squamous cell carcinoma a...,Primary intraosseous carcinoma (P1OC) of the j...
4,20699517,Inflammatory myofibroblastic tumor parieto-occ...,Inflammatory myofibroblastic tumor is a divers...


In [7]:
# make all IDs strings, just to avoid annoying mismatches
train_qrels["query_id"] = train_qrels["query_id"].astype(str)
train_qrels["corpus_id"] = train_qrels["corpus_id"].astype(str)

patient_info["_id"] = patient_info["_id"].astype(str)
articles["_id"] = articles["_id"].astype(str)

print(train_qrels.columns.tolist())
print(patient_info.columns.tolist())
print(articles.columns.tolist())

['query_id', 'corpus_id', 'score']
['_id', 'text']
['_id', 'title', 'text']


In [8]:
#### filter qrels so only top article row remains for each patient

first_article_links = train_qrels.drop_duplicates(subset="query_id", keep="first").copy()

print("Rows in original qrels:", len(train_qrels))
print("Rows after taking first article per patient:", len(first_article_links))

display(first_article_links.head())

Rows in original qrels: 742718
Rows after taking first article per patient: 54643


,query_id,corpus_id,score
0,8674405-1,29510273,1
13,8674685-1,30281871,1
21,8675572-1,21380941,1
24,8675577-1,25765179,1
42,8675583-1,30694442,1


In [9]:
#### merge on the patient text

patient_article_pairs = first_article_links.merge(
    patient_info,
    left_on="query_id",
    right_on="_id",
    how="left"
)

print("After merging patient info:", patient_article_pairs.shape)
display(patient_article_pairs.head())


#### merge on article title and abstract

patient_article_pairs = patient_article_pairs.merge(
    articles,
    left_on="corpus_id",
    right_on="_id",
    how="left",
    suffixes=("_patient", "_article")
)

print("After merging articles:", patient_article_pairs.shape)
display(patient_article_pairs.head())

After merging patient info: (54643, 5)


,query_id,corpus_id,score,_id,text
0,8674405-1,29510273,1,8674405-1,A 36-year-old G4P2 premenopausal woman with a ...
1,8674685-1,30281871,1,8674685-1,"We report a case of a 45-year-old woman, a non..."
2,8675572-1,21380941,1,8675572-1,A 67-year-old Caucasian female presented to ou...
3,8675577-1,25765179,1,8675577-1,Our patient is a 78-year-old male with a past ...
4,8675583-1,30694442,1,8675583-1,A 64-year-old Caucasian male smoker with a hor...


After merging articles: (54643, 8)


,query_id,corpus_id,score,_id_patient,text_patient,_id_article,title,text_article
0,8674405-1,29510273,1,8674405-1,A 36-year-old G4P2 premenopausal woman with a ...,29510273,Ovarian suppression failure during GnRH agonis...,In premenopausal women treated for breast canc...
1,8674685-1,30281871,1,8674685-1,"We report a case of a 45-year-old woman, a non...",30281871,Langerhans cell histiocytosis in adults: Advan...,Langerhans cell histiocytosis (LCH) is a rare ...
2,8675572-1,21380941,1,8675572-1,A 67-year-old Caucasian female presented to ou...,21380941,Colorectal cancer screening in women with endo...,Colorectal cancer (CRC) is the most common gas...
3,8675577-1,25765179,1,8675577-1,Our patient is a 78-year-old male with a past ...,25765179,Somatostatin Receptors 2A and 5 Are Expressed ...,Merkel cell carcinoma (MCC) is a rare high-gra...
4,8675583-1,30694442,1,8675583-1,A 64-year-old Caucasian male smoker with a hor...,30694442,The Challenge of Managing Bladder Cancer and U...,Bladder cancer is the fourth most common cance...


In [10]:
#### inspect column names, rename 

print(patient_article_pairs.columns.tolist())

final_pairs = patient_article_pairs[
    [
        "query_id",
        "corpus_id",
        "text_patient",
        "title",
        "text_article"
    ]
].copy()

final_pairs = final_pairs.rename(columns={
    "query_id": "patient_id",
    "corpus_id": "article_id",
    "text_patient": "patient_text",
    "text_article": "article_text"
})

display(final_pairs.head())

['query_id', 'corpus_id', 'score', '_id_patient', 'text_patient', '_id_article', 'title', 'text_article']


,patient_id,article_id,patient_text,title,article_text
0,8674405-1,29510273,A 36-year-old G4P2 premenopausal woman with a ...,Ovarian suppression failure during GnRH agonis...,In premenopausal women treated for breast canc...
1,8674685-1,30281871,"We report a case of a 45-year-old woman, a non...",Langerhans cell histiocytosis in adults: Advan...,Langerhans cell histiocytosis (LCH) is a rare ...
2,8675572-1,21380941,A 67-year-old Caucasian female presented to ou...,Colorectal cancer screening in women with endo...,Colorectal cancer (CRC) is the most common gas...
3,8675577-1,25765179,Our patient is a 78-year-old male with a past ...,Somatostatin Receptors 2A and 5 Are Expressed ...,Merkel cell carcinoma (MCC) is a rare high-gra...
4,8675583-1,30694442,A 64-year-old Caucasian male smoker with a hor...,The Challenge of Managing Bladder Cancer and U...,Bladder cancer is the fourth most common cance...


In [ ]:
#### check for mising merges

print("Missing patient text:", final_pairs["patient_text"].isna().sum())
print("Missing article text:", final_pairs["article_text"].isna().sum())
print("Missing article tisle:", final_pairs["title"].isna().sum())

missing_rows = final_pairs[
    final_pairs["patient_text"].isna() |
    final_pairs["article_text"].isna()
]

Missing patient text: 0
Missing article text: 0
Missing article tisle: 0


In [12]:
# write final_pairs into separate file

records = final_pairs.to_dict(orient = "records")
srsly.write_jsonl("merged_PAR_train.jsonl", records)

**Execute starting here**

In [3]:
# read file that so that previous code does not need to be rerun

merged_pairs = pd.DataFrame(list(srsly.read_jsonl("merged_PAR_train.jsonl")))  


In [14]:
display(merged_pairs.head())

,patient_id,article_id,patient_text,title,article_text
0,8674405-1,29510273,A 36-year-old G4P2 premenopausal woman with a ...,Ovarian suppression failure during GnRH agonis...,In premenopausal women treated for breast canc...
1,8674685-1,30281871,"We report a case of a 45-year-old woman, a non...",Langerhans cell histiocytosis in adults: Advan...,Langerhans cell histiocytosis (LCH) is a rare ...
2,8675572-1,21380941,A 67-year-old Caucasian female presented to ou...,Colorectal cancer screening in women with endo...,Colorectal cancer (CRC) is the most common gas...
3,8675577-1,25765179,Our patient is a 78-year-old male with a past ...,Somatostatin Receptors 2A and 5 Are Expressed ...,Merkel cell carcinoma (MCC) is a rare high-gra...
4,8675583-1,30694442,A 64-year-old Caucasian male smoker with a hor...,The Challenge of Managing Bladder Cancer and U...,Bladder cancer is the fourth most common cance...


In [15]:
only_text = merged_pairs.drop(merged_pairs.columns[[0, 1]], axis = 1)

In [16]:
only_text.head()

,patient_text,title,article_text
0,A 36-year-old G4P2 premenopausal woman with a ...,Ovarian suppression failure during GnRH agonis...,In premenopausal women treated for breast canc...
1,"We report a case of a 45-year-old woman, a non...",Langerhans cell histiocytosis in adults: Advan...,Langerhans cell histiocytosis (LCH) is a rare ...
2,A 67-year-old Caucasian female presented to ou...,Colorectal cancer screening in women with endo...,Colorectal cancer (CRC) is the most common gas...
3,Our patient is a 78-year-old male with a past ...,Somatostatin Receptors 2A and 5 Are Expressed ...,Merkel cell carcinoma (MCC) is a rare high-gra...
4,A 64-year-old Caucasian male smoker with a hor...,The Challenge of Managing Bladder Cancer and U...,Bladder cancer is the fourth most common cance...


In [35]:
### prepare LLM
BASE_URL = "https://llm1-compute.cms.hu-berlin.de/v1/"
OPENAI_API_KEY = "secret_but_not_used"
PROMPT_FILE = "hippo_questions_prompt.txt"

with open(PROMPT_FILE, "r", encoding="utf-8") as f:
    PROMPT_TEMPLATE = f.read()

INPUT_FILE = "merged_PAR_train.jsonl"
OUTPUT_FILE_JSON = 'questions.jsonl'
OUTPUT_FILE_TXT = 'questions.txt'

client = OpenAI(base_url=BASE_URL, api_key=OPENAI_API_KEY)

In [44]:
### specify output format
class QuestionResponse(BaseModel):
    question: str
    explanation: str = Field(..., description="A very brief 2-sentence summary of the medical link.")

In [45]:
def generate_question(patient_text, article_title, article_text):
    prompt = PROMPT_TEMPLATE.format(
        patient_text=patient_text or "",
        article_title=article_title or "",
        article_text=article_text or ""
    )

    try:
        response = client.beta.chat.completions.parse(
            model="llm1",
            messages=[
                {"role": "system", "content": "You are a medical researcher. Output a single JSON object representing the best possible question and its explanation. The explanation must be a single paragraph of maximum 40 words."},
                {"role": "user", "content": prompt}
            ],
            response_format=QuestionResponse,
            temperature=0.1
        )
        return response.choices[0].message.parsed
    except APIConnectionError as e:
        return f"Error connecting to API: {e}"
    except Exception as e:
        return f"An error occurred: {e}"
    

In [ ]:
### let llm generate questions
with open(INPUT_FILE, 'r', encoding='utf-8') as f_in, \
     open(OUTPUT_FILE_JSON, 'w', encoding='utf-8') as f_out_json, \
     open(OUTPUT_FILE_TXT, 'w', encoding='utf-8') as f_out_txt:

    lines = f_in.readlines()[:10] 

    for line in lines:
        data = json.loads(line)
        
        patient_id = data.get('patient_id')
        patient_text = data.get('patient_text')
        article_id = data.get('article_id')
        article_title = data.get('title')
        article_text = data.get('article_text')
    
        response = generate_question(patient_text, article_title, article_text)

        result = {
            "patient_id": patient_id,
            "article_id": article_id,
            "question": response.question,
            "explanation": response.explanation
        }

        f_out_json.write(json.dumps(result, ensure_ascii=False) + '\n')
        
        # TXT format
        f_out_txt.write(f"---------------------------------------- PATIENT ID: {patient_id} ----------------------------------------\n")
        f_out_txt.write(f"Article: {article_title}\n")
        f_out_txt.write(f"Question:\n{response.question}\n")
        f_out_txt.write(f"Explanation:\n{response.explanation}\n\n")
    
        f_out_json.flush()
        f_out_txt.flush()
